# Python 原生批量向量查询（IVF + RaBitQ）

本教程直接调用 `session.batch_search()` 和 `table.search_batch()`，输入为 NumPy 二维数组，不通过 SQL 表函数。它演示 L2/IP、index/exact、结果结构、小数据集基线与一次完整调用的计时，以及索引/表/目录清理。

示例使用256个16维向量、300个独立query，每次仅保留一个演示表；300个query可覆盖内部多window执行。它不是大规模性能基准。

前置：已初始化的PostgreSQL元数据库、带`ci-unify`的LakeSoul Python扩展、NumPy和PyArrow。元数据库由`LAKESOUL_PG_URL`、`LAKESOUL_PG_USERNAME`、`LAKESOUL_PG_PASSWORD`配置，本Notebook不会初始化或清空整个元数据库。

请使用**新kernel从第一格运行**；第一格在import之前配置独立缓存。清理后如需再次运行，应重启kernel。中途失败也应执行最后的清理单元。

SQL版本另见 [原生批量SQL Notebook](native_batch_sql.ipynb)。

In [1]:
import os
import sys
import uuid
import tempfile
import shutil
from pathlib import Path
from time import perf_counter

if "lakesoul" in sys.modules:
    raise RuntimeError("请使用新kernel从第一格运行；缓存配置须在首次导入/查询前设置。")

run_id = "native_python_batch_" + uuid.uuid4().hex
root = Path(tempfile.mkdtemp(prefix=run_id + "_")).resolve()
(root / ".owner").write_text(run_id)
os.environ.update({
    "LAKESOUL_VECTOR_CACHE": "1",
    "LAKESOUL_VECTOR_MEMORY_BYTES": "512MiB",
    "LAKESOUL_CACHE": "1",
    "LAKESOUL_CACHE_SIZE": "1GiB",
    "LAKESOUL_CACHE_PATH": str(root / "cache" / "data"),
    "LAKESOUL_INDEX_SSD_BYTES": "0",
})

import numpy as np
import pyarrow as pa
from lakesoul import LakeSoulCatalog
from lakesoul.exceptions import TableNotFoundError

catalog = LakeSoulCatalog.from_env(namespace="default")
session = catalog.session(reuse=False)
owned_tables = {}
print("本次演示owner:", run_id)
print("独立演示目录:", root)


本次演示owner: native_python_batch_a505cf14629d441d882a122915b8e62f
独立演示目录: /tmp/native_python_batch_a505cf14629d441d882a122915b8e62f_6gsgqh3r


## 1. 创建满足原生路径条件的小表

使用单列Int64主键`id`，声明16维、4个IVF cluster、7-bit RaBitQ索引；一次写入后自动建索引。目标无CDC/mutation chain，索引完整覆盖数据文件。

下面的辅助函数只创建或删除本次owner的表和本地目录；L2完成后先清理，再创建IP表。

In [2]:
DIM, NBASE, NQUERY, TOP_K, NLIST = 16, 256, 300, 5, 4
PARALLELISM, OVERFETCH = 2, 64
rng = np.random.default_rng(2026)
base = rng.normal(size=(NBASE, DIM)).astype(np.float32)
queries = rng.normal(size=(NQUERY, DIM)).astype(np.float32)
vector_type = pa.list_(pa.field("item", pa.float32()), DIM)
schema = pa.schema([
    pa.field("id", pa.int64(), False),
    pa.field("vec", vector_type, False),
])

def assert_owner_root():
    assert not root.is_symlink()
    assert root.resolve().parent == Path(tempfile.gettempdir()).resolve()
    assert root.name.startswith(run_id + "_")
    assert (root / ".owner").read_text() == run_id

def create_demo_table(suffix, metric, vectors):
    assert_owner_root()
    assert not owned_tables, "先清理上一张演示表，再创建下一张"
    name = run_id + "_" + suffix
    path = root / suffix
    table = catalog.create_table(
        name, path=path.as_uri(), schema=schema,
        primary_keys=["id"], hash_bucket_num=4,
        properties={"native_python_batch_owner": run_id},
        vector_index=[{"column": "vec", "dim": DIM, "nlist": NLIST,
                       "total_bits": 7, "metric": metric}],
    )
    owned_tables[name] = path
    batch = pa.RecordBatch.from_arrays([
        pa.array(np.arange(len(vectors), dtype=np.int64)),
        pa.FixedSizeListArray.from_arrays(pa.array(vectors.reshape(-1)), DIM),
    ], schema=schema)
    table.write_arrow(batch, format="vortex", auto_build_vector_index=True)
    return table

def drop_demo_table(name):
    assert_owner_root()
    path = owned_tables[name]
    assert path.parent == root and not path.is_symlink()
    try:
        table = catalog.table(name)
    except TableNotFoundError:
        table = None
    if table is not None:
        assert str(table.path).rstrip("/") == path.as_uri().rstrip("/")
        assert table.properties.get("native_python_batch_owner") == run_id
        report = table.drop_vector_index(
            column="vec", purge_files=True, if_exists=True,
            grace_seconds=0, wait_for_cleanup=True,
        )
        catalog.drop_table(name)
        print("已删除演示索引shards:", report["shards_dropped"])
    if path.exists():
        resolved = path.resolve()
        assert resolved.parent == root.resolve() and resolved == path
        shutil.rmtree(resolved)
    assert not path.exists()
    del owned_tables[name]

def cleanup_demo():
    session.close()
    for name in list(owned_tables):
        drop_demo_table(name)
    assert_owner_root()
    resolved = root.resolve()
    assert resolved.parent == Path(tempfile.gettempdir()).resolve()
    shutil.rmtree(resolved)
    assert not root.exists()

l2_table = create_demo_table("l2_items", "L2", base)
print("L2表:", l2_table.name, "base:", base.shape, "queries:", queries.shape)


L2表: native_python_batch_a505cf14629d441d882a122915b8e62f_l2_items base: (256, 16) queries: (300, 16)


## 2. 直接调用原生会话批入口：index模式

`session.batch_search()`直接进入Rust批查询入口；每次传入整批query，返回每query一份邻居ID列表。它不会在Python层因错误而逐query循环SQL，原生调用失败会抛出异常。

本例显式设置`parallelism=2`；实际并行度仍受共享CPU池上限限制。`batch_size=0`交给实现选择，内部使用不超过256个query的有界window，不需要用户手动循环单查。

In [3]:
l2_index = session.batch_search(
    l2_table.name, queries, namespace="default", column="vec",
    top_k=TOP_K, nprobe=NLIST, metric="L2", exact=False,
    batch_size=0, parallelism=PARALLELISM,
)
assert len(l2_index) == NQUERY
assert all(len(ids) == TOP_K and len(set(ids)) == TOP_K for ids in l2_index.ids)
assert all(0 <= key < NBASE for ids in l2_index.ids for key in ids)
assert l2_index.distances is None
print("返回query组数:", len(l2_index))
print("前3个query的index邻居:", l2_index.ids[:3])


返回query组数: 300
前3个query的index邻居: [[240, 66, 149, 252, 166], [234, 189, 41, 232, 237], [130, 70, 217, 62, 64]]


## 3. exact重排与正确计时

`exact=True`读取ANN候选的原始向量重排，不等于一般情况下保证全库精确kNN。

为了让这个小例子能与完整暴力基线逐项比较，探测全部4个cluster，并将`exact_overfetch=64`设为足以覆盖全部256行；这是验证设置，生产需要按recall/耗时调节。

计时只围绕**一次完整原生调用**，基线计算、结果校验都在计时外。以下小数据QPS只演示计算方法，不能作为大规模性能结论。

In [4]:
started = perf_counter()
l2_exact = session.batch_search(
    l2_table.name, queries, namespace="default", column="vec",
    top_k=TOP_K, nprobe=NLIST, metric="L2", exact=True,
    exact_overfetch=OVERFETCH, batch_size=0, parallelism=PARALLELISM,
)
wall_seconds = perf_counter() - started

# 小数据集的完整基线；这一段不在原生调用计时内。
l2_squared = np.sum((queries[:, None, :] - base[None, :, :]) ** 2, axis=2)
l2_truth = np.argsort(l2_squared, axis=1, kind="stable")[:, :TOP_K]
assert l2_exact.ids == l2_truth.tolist()
assert len(l2_exact) == NQUERY and l2_exact.distances is None

def recall_at_k(ids, expected):
    return float(np.mean([
        len(set(actual) & set(truth)) / TOP_K
        for actual, truth in zip(ids, expected)
    ]))

print(f"一次完整调用: {NQUERY} queries / {wall_seconds:.6f} 秒")
print(f"演示QPS: {NQUERY / wall_seconds:.1f}")
print("index recall@5:", recall_at_k(l2_index.ids, l2_truth))
print("exact recall@5:", recall_at_k(l2_exact.ids, l2_truth))


一次完整调用: 300 queries / 0.041681 秒
演示QPS: 7197.5
index recall@5: 0.8699999999999998
exact recall@5: 1.0


## 4. 表级便捷入口与返回结构

`table.search_batch(..., rerank="exact")`将参数转给批查询入口。它为不支持的表保留逐query SQL兼容回退；需要严格使用原生路径、希望错误直接暴露时，使用上面的`session.batch_search()`。

当前Python批接口只返回IDs，`distances`为`None`；即使exact也不会自动增加距离列。`to_arrow()`返回`query_id, rank, id`，其中**Python的query_id和rank都从0开始**。SQL批函数的rank从1开始，比较结果时注意这一点。

In [5]:
l2_via_table = l2_table.search_batch(
    queries, column="vec", top_k=TOP_K, metric="L2",
    rerank="exact", nprobe=NLIST, exact_overfetch=OVERFETCH,
    batch_size=0, parallelism=PARALLELISM, session=session,
)
assert l2_via_table.ids == l2_exact.ids
rows = l2_via_table.to_arrow()
assert rows.num_rows == NQUERY * TOP_K
assert rows.column_names == ["query_id", "rank", "id"]
assert rows.column("query_id").to_pylist()[:TOP_K] == [0] * TOP_K
assert rows.column("rank").to_pylist()[:TOP_K] == list(range(TOP_K))
print("表级入口与直接原生入口的IDs一致")
rows.slice(0, 3 * TOP_K)


表级入口与直接原生入口的IDs一致
pyarrow.Table
query_id: int64
rank: int64
id: int64
----
query_id: [[0,0,0,0,0,...,2,2,2,2,2]]
rank: [[0,1,2,3,4,...,0,1,2,3,4]]
id: [[240,66,149,1,126,...,70,130,62,64,196]]


## 5. 先释放L2索引和表目录

查询已完整返回，才删除该演示表的索引、元数据及目录。这里不删除其它表，也不复用本次表名；之后才创建IP演示表。

In [6]:
drop_demo_table(l2_table.name)
assert owned_tables == {}
print("L2索引、表及目录已释放；接下来创建IP表")


已删除演示索引shards: 4
L2索引、表及目录已释放；接下来创建IP表


## 6. 原生IP批查询

IP按内积从大到小选邻居。本例先将base/query归一化，因此其邻居排名也对应cosine相似度；不归一化时仍是普通内积检索。

两种Python入口同样支持IP的index和exact。exact与小数据集的完整负内积排序基线比较；Python结果仍只有IDs。

In [7]:
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    assert np.all(norms > 0)
    return (vectors / norms).astype(np.float32)

ip_base, ip_queries = normalize(base), normalize(queries)
ip_table = create_demo_table("ip_items", "IP", ip_base)
ip_index = session.batch_search(
    ip_table.name, ip_queries, namespace="default", column="vec",
    top_k=TOP_K, nprobe=NLIST, metric="IP", exact=False,
    batch_size=0, parallelism=PARALLELISM,
)
ip_exact = session.batch_search(
    ip_table.name, ip_queries, namespace="default", column="vec",
    top_k=TOP_K, nprobe=NLIST, metric="IP", exact=True,
    exact_overfetch=OVERFETCH, batch_size=0, parallelism=PARALLELISM,
)
ip_truth = np.argsort(-(ip_queries @ ip_base.T), axis=1, kind="stable")[:, :TOP_K]
assert ip_exact.ids == ip_truth.tolist()
assert len(ip_index) == len(ip_exact) == NQUERY
assert all(len(ids) == TOP_K for ids in ip_index.ids)
assert ip_exact.distances is None
ip_via_table = ip_table.search_batch(
    ip_queries, column="vec", metric="IP", top_k=TOP_K,
    rerank="exact", nprobe=NLIST, exact_overfetch=OVERFETCH,
    batch_size=0, parallelism=PARALLELISM, session=session,
)
assert ip_via_table.ids == ip_exact.ids
print("IP index recall@5:", recall_at_k(ip_index.ids, ip_truth))
print("IP exact recall@5:", recall_at_k(ip_exact.ids, ip_truth))
print("前3个query的IP exact邻居:", ip_exact.ids[:3])


IP index recall@5: 0.8513333333333332
IP exact recall@5: 1.0
前3个query的IP exact邻居: [[240, 129, 66, 252, 24], [234, 189, 10, 41, 232], [183, 130, 207, 10, 70]]


## 7. 空输入与维度错误

空query列表返回空结果。维度不匹配必须报错，不能把缺失结果当作成功。

In [8]:
empty = session.batch_search(
    ip_table.name, [], namespace="default", column="vec", metric="IP",
)
assert len(empty) == 0 and empty.ids == []
try:
    ip_table.search_batch(
        ip_queries[:, :-1], column="vec", top_k=TOP_K,
        metric="IP", rerank="index", session=session,
    )
except ValueError as error:
    print("预期的维度错误:", error)
else:
    raise AssertionError("错误维度应被拒绝")
print("空输入与错误输入处理通过")


预期的维度错误: query dimension 15 does not match the index dimension 16
空输入与错误输入处理通过


## 8. 最终清理

即使前面单元失败，也请执行本单元。只处理本次owner的演示表、索引和临时目录，包括独立cache；不清空元数据库。执行后如需重新运行，请重启kernel。

完整执行路径和适用边界见 [原生批查询说明](../native-batch-sql.md)。本教程的主路径是Python直接原生入口；需要批量距离列、外部qid或query表输入时，可以使用 [SQL批入口](native_batch_sql.ipynb)。

In [9]:
cleanup_demo()
assert owned_tables == {} and not root.exists()
print("已清理本次Python演示的索引、表、数据及独立缓存")


已删除演示索引shards: 4
已清理本次Python演示的索引、表、数据及独立缓存
